# 02. RadixAttention 与 Prefix KV Cache

> 面向即将加入 SGLang 开源社区的开发者。建议从仓库根目录启动 Jupyter，
> 或者在 notebook 第一格运行路径检查。本文以本地源码为主，线上文档为索引。
> Markdown 中的源码摘录会插入 `# 注：...` 行内讲解；可执行代码 cell 则保持可运行。


In [ ]:
from pathlib import Path
import ast, inspect, re, textwrap

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "python" / "sglang").exists() and (candidate / "docs").exists():
            return candidate
    raise RuntimeError("没有找到 SGLang 仓库根目录，请从仓库内启动 notebook。")

ROOT = find_repo_root()
print(ROOT)

def read_rel(path):
    return (ROOT / path).read_text()

def show_lines(path, start, end):
    lines = read_rel(path).splitlines()
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:4d}: {lines[i-1]}")

def find_defs(path, names=None):
    tree = ast.parse(read_rel(path))
    rows = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            if names is None or node.name in names:
                rows.append((node.lineno, type(node).__name__, node.name))
    return sorted(rows)


## 先纠正一个常见误解

`RadixAttention` 这个名字容易让人以为它只是一个 attention kernel。实际上它是一个跨层设计：

- `RadixAttention` layer 是模型中的 attention 抽象，负责把 Q/K/V 与 `ForwardBatch` 交给当前 attention backend。
- `RadixCache` / `HiRadixCache` 是 prefix KV cache 的 metadata tree，负责匹配、插入、拆分、锁定、驱逐。
- `SchedulePolicy` 会利用 prefix hit 长度调整 waiting queue 的优先级。
- KV memory pool 负责把 logical token span 映射到实际 KV slot。

RadixAttention 的收益来自“相同前缀不重新 prefill”，而不是仅仅来自某个单独 kernel。


In [ ]:
for path in [
    "python/sglang/srt/layers/radix_attention.py",
    "python/sglang/srt/mem_cache/radix_cache.py",
    "python/sglang/srt/managers/schedule_policy.py",
]:
    print("\n==", path)
    for row in find_defs(path, names={"RadixAttention", "RadixCache", "RadixKey", "TreeNode", "match_prefix_for_req", "SchedulePolicy"}):
        print(row)


## `RadixKey`：prefix matching 的基本单位

`RadixKey` 包装 token ids，同时带 `extra_key`。`extra_key` 很关键：同样 token 序列在不同 LoRA、cache salt、某些隔离策略下不能混用 KV。

它还支持 bigram view，这是 EAGLE 这类 speculative decoding 与 prefix cache 交互时会用到的特殊视图。


### `RadixKey` 的字段：token 序列之外还要带隔离维度

```python
# python/sglang/srt/mem_cache/radix_cache.py:35-75
  35: logger = logging.getLogger(__name__)
  36: 
  37: from sglang.srt.mem_cache.base_prefix_cache import (
  38:     BasePrefixCache,
  39:     DecLockRefParams,
  40:     DecLockRefResult,
  41:     EvictParams,
  42:     EvictResult,
  43:     IncLockRefResult,
  44:     InsertParams,
  45:     InsertResult,
  46:     MatchPrefixParams,
  47:     MatchResult,
  48: )
  49: from sglang.srt.mem_cache.events import KVCacheEventMixin
  50: from sglang.srt.mem_cache.session_radix_cache import SessionRadixCacheMixin
  51: from sglang.srt.mem_cache.utils import get_eviction_strategy, split_node_hash_value
  52: 
  53: if TYPE_CHECKING:
      # 注：类型检查分支：只给静态类型工具导入重依赖，运行时不会进入这条路径。
  54:     from sglang.srt.managers.schedule_batch import Req
  55: 
  56: 
  57: class RadixKey:
      # 注：类定义：`RadixKey` 是这一段的状态/行为边界；先看字段，再看哪些方法会改字段。
  58:     """is_bigram=True: token_ids holds raw tokens (N+1 for N bigrams); slices share one boundary token."""
  59: 
  60:     __slots__ = ("token_ids", "extra_key", "is_bigram", "limit")
      # 注：成员变量声明：`__slots__` 限定实例可保存的字段，读这个类时可先把这些字段当作对象状态地图。
  61: 
  62:     def __init__(
      # 注：函数定义：`__init__` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  63:         self,
  64:         token_ids: array[int],
  65:         extra_key: Optional[str] = None,
  66:         is_bigram: bool = False,
  67:         limit: Optional[int] = None,
  68:     ):
  69:         # token ids sequence (raw ints in both modes)
  70:         self.token_ids = token_ids
      # 注：成员变量写入：`self.token_ids` 保存 token 相关状态，后续长度计算、cache key 或采样都会读取它。
  71:         # extra key (e.g. lora_id, cache_salt)
  72:         self.extra_key = extra_key
      # 注：成员变量写入：`self.extra_key` 会留在对象生命周期中，后续方法可能依赖这份状态。
  73:         # bigram view over token_ids: length = max(0, len(token_ids) - 1)
  74:         self.is_bigram = is_bigram
      # 注：成员变量写入：`self.is_bigram` 会留在对象生命周期中，后续方法可能依赖这份状态。
  75:         # Optional cap on raw tokens: behave as if token_ids were sliced to
```

**读这段时抓住：**

- `token_ids` 是 radix tree 的路径内容，但 `extra_key` 才决定这段 KV 能不能跨请求复用。
- `is_bigram` 让同一份 token array 可以被解释为 bigram 序列，避免为 EAGLE 等路径额外 materialize tuple list。
- `limit` 是 O(1) 视图上限，用来避免为了截断前缀而复制长 token 序列。


### `RadixKey.match`：长公共前缀用 galloping search 避免 Python 逐 token 扫

```python
# python/sglang/srt/mem_cache/radix_cache.py:120-160
 120:             raise ValueError("RadixKey slice step must be 1")
      # 注：函数/库调用：`ValueError` 把当前阶段委托给外部 helper 或库实现。
 121: 
 122:         if self.is_bigram:
      # 注：分支判断：只有满足 `self.is_bigram` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 123:             # bigrams [start, stop) span raw tokens [start, stop + 1);
 124:             # empty slice -> empty raw tokens (not a dangling boundary token).
 125:             raw = self.token_ids[start : stop + 1] if stop > start else array("q")
      # 注：局部状态绑定：`raw` 保存本阶段的中间结果，后续几行通常会立即消费它。
 126:             return RadixKey(raw, self.extra_key, is_bigram=True)
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
 127:         return RadixKey(self.token_ids[start:stop], self.extra_key)
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
 128: 
 129:     def __repr__(self) -> str:
      # 注：函数定义：`__repr__` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 130:         preview = self.token_ids[:10]
      # 注：局部状态绑定：`preview` 保存本阶段的中间结果，后续几行通常会立即消费它。
 131:         return f"RadixKey(extra_key={self.extra_key!r}, token_ids={preview}{'...' if len(self.token_ids) > 10 else ''}, is_bigram={self.is_bigram})"
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
 132: 
 133:     def page_aligned(self, page_size: int) -> RadixKey:
      # 注：函数定义：`page_aligned` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 134:         if page_size == 1:
      # 注：分支判断：只有满足 `page_size == 1` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 135:             return self
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
 136:         aligned_len = len(self) // page_size * page_size
      # 注：局部状态绑定：`aligned_len` 保存本阶段的中间结果，后续几行通常会立即消费它。
 137:         return self[:aligned_len]
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
 138: 
 139:     def maybe_to_bigram_view(
      # 注：函数定义：`maybe_to_bigram_view` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 140:         self,
 141:         is_eagle: bool,
 142:         value: Optional[torch.Tensor] = None,
 143:     ) -> Tuple[RadixKey, Optional[torch.Tensor]]:
 144:         # O(1): flip the bigram flag instead of materializing a tuple list.
 145:         # value is paired with raw tokens and gets truncated to the bigram count.
 146:         if is_eagle and not self.is_bigram:
      # 注：分支判断：只有满足 `is_eagle and not self.is_bigram` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 147:             self.is_bigram = True
      # 注：成员变量写入：`self.is_bigram` 会留在对象生命周期中，后续方法可能依赖这份状态。
 148:             if value is not None:
      # 注：分支判断：只有满足 `value is not None` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 149:                 value = value[: len(self)]
      # 注：局部状态绑定：`value` 保存本阶段的中间结果，后续几行通常会立即消费它。
 150:         return self, value
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
 151: 
 152:     def _check_compatible(self, other: RadixKey) -> None:
      # 注：函数定义：`_check_compatible` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 153:         if self.extra_key != other.extra_key:
      # 注：分支判断：只有满足 `self.extra_key != other.extra_key` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 154:             raise ValueError(
      # 注：函数/库调用：`ValueError` 把当前阶段委托给外部 helper 或库实现。
 155:                 f"RadixKey operations require matching extra_key, but got "
 156:                 f"{self.extra_key=} != {other.extra_key=}"
 157:             )
 158: 
 159:     def match(self, other: RadixKey, page_size: int = 1) -> int:
      # 注：函数定义：`match` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 160:         """Logical-unit prefix length shared with ``other``. Result is rounded down to ``page_size``."""
```

**读这段时抓住：**

- 先检查 `extra_key`，这是 cache correctness 的第一道闸。
- 指数窗口 + 二分定位第一个分叉点，优化的是长 shared prefix 场景，这正是 prefix cache 的热点。
- `page_size` 对齐在这里完成；后续返回的 KV indices 才能安全对应 page allocator。


In [ ]:
from array import array
from sglang.srt.mem_cache.radix_cache import RadixKey

a = RadixKey(array("q", [1, 2, 3, 4, 5]), extra_key="tenant-a")
b = RadixKey(array("q", [1, 2, 3, 9]), extra_key="tenant-a")
c = RadixKey(array("q", [1, 2, 3, 4]), extra_key="tenant-b")

print("a vs b match:", a.match(b))
try:
    print(a.match(c))
except ValueError as e:
    print("extra_key isolation:", e)

bigram = RadixKey(array("q", [10, 11, 12, 13]), is_bigram=True)
print("bigram logical units:", list(bigram))
print("bigram len:", len(bigram))


## `RadixCache` 的核心操作

需要抓住四个操作：

- `match_prefix`：沿 radix tree 找最长已缓存前缀，返回 device indices、last node、host hit 等信息。
- `insert`：把完成 prefill/decode 的 KV span 插入 tree；必要时拆分节点。
- `inc_lock_ref` / `dec_lock_ref`：保护正在被请求使用的节点，避免被驱逐。
- `evict`：按策略回收未锁定叶子节点，释放 KV slots。


In [ ]:
for name in ["match_prefix", "insert", "cache_finished_req", "cache_unfinished_req", "evict", "inc_lock_ref", "dec_lock_ref", "_split_node"]:
    rows = find_defs("python/sglang/srt/mem_cache/radix_cache.py", {name})
    print(rows[0] if rows else "missing", name)


In [ ]:
show_lines("python/sglang/srt/mem_cache/radix_cache.py", 360, 455)


### `RadixCache.match_prefix`：返回的不只是长度，而是一组调度/加载锚点

```python
# python/sglang/srt/mem_cache/radix_cache.py:360-418
 360:     def match_prefix(self, params: MatchPrefixParams) -> MatchResult:
      # 注：函数定义：`match_prefix` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 361:         """Find the longest cached prefix of ``key`` in the radix tree.
 362: 
 363:         The logical namespace for prefix matching is determined by both the
 364:         token id sequence and the optional ``extra_key`` carried by ``RadixKey``.
 365:         Entries that share identical leading token ids but have *different*
 366:         ``extra_key`` values are intentionally kept disjoint and never share
 367:         prefix nodes. This is useful to:
 368: 
 369:         * Isolate KV cache lines for different LoRA / adapter IDs.
 370:         * Separate requests that intentionally should not share state (e.g.,
 371:           different sampling salt, cache version, or retrieval augmentation
 372:           context) by supplying a distinct ``extra_key``.
 373: 
 374:         Args:
 375:             params (MatchPrefixParams): Parameters containing the lookup key
 376:                 with a list of token ids and an optional ``extra_key`` namespace tag.
 377:                 If ``page_size > 1`` the length is internally truncated to a multiple
 378:                 of ``page_size`` before matching. Passing an empty key returns an
 379:                 empty result with the root as the last node.
 380: 
 381:         Returns:
 382:             MatchResult: ``device_indices`` is a 1-D ``torch.int64`` tensor of
      # 注：字段/变量声明：`MatchResult` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 383:             the concatenated KV cache indices corresponding to the longest
 384:             cached prefix (may be length 0).
 385:             ``last_device_node`` and ``last_host_node`` (currently the same) are the tree node objects
 386:             representing the terminal node of the matched prefix. This method
 387:             may mutate internal structure by splitting an existing node if the
 388:             match ends inside a stored segment.
 389: 
 390:         Internal updates:
 391:             * Refreshes access metadata (timestamps) used by the
 392:                 configured eviction strategy.
 393:             * If the lookup ends inside a stored segment the node is split once
 394:                 to expose a precise boundary; this structural refinement improves
 395:                 subsequent match efficiency and does not duplicate data.
 396:         """
 397:         key = params.key
      # 注：局部状态绑定：`key` 保存本阶段的中间结果，后续几行通常会立即消费它。
 398:         key, _ = key.maybe_to_bigram_view(self.is_eagle)
      # 注：局部状态绑定：`key, _` 保存本阶段的中间结果，后续几行通常会立即消费它。
 399: 
 400:         if self.disable or len(key) == 0:
      # 注：分支判断：只有满足 `self.disable or len(key) == 0` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 401:             return self._empty_match_result
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
 402: 
 403:         key = key.page_aligned(self.page_size)
      # 注：局部状态绑定：`key` 接住关键调用结果，后续代码会基于它继续装配组件或推进请求。
 404: 
 405:         if len(key) == 0:
      # 注：分支判断：只有满足 `len(key) == 0` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 406:             return self._empty_match_result
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
 407: 
 408:         value, last_node = self._match_prefix_helper(self.root_node, key)
      # 注：局部状态绑定：`value, last_node` 接住关键调用结果，后续代码会基于它继续装配组件或推进请求。
 409:         if value:
      # 注：分支判断：只有满足 `value` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 410:             value = torch.cat(value)
      # 注：局部状态绑定：`value` 保存本阶段的中间结果，后续几行通常会立即消费它。
 411:         else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
 412:             value = self._empty_match_result.device_indices
      # 注：局部状态绑定：`value` 保存本阶段的中间结果，后续几行通常会立即消费它。
 413:         return MatchResult(
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
 414:             device_indices=value,
 415:             last_device_node=last_node,
 416:             last_host_node=last_node,
 417:             best_match_node=last_node,
 418:         )
```

**读这段时抓住：**

- `key.page_aligned` 说明 cache hit 的最小合法单位受 page size 约束。
- `value` 是已命中的 device KV slot indices；scheduler 后续会把它放进 `req.prefix_indices`。
- `last_device_node` / `last_host_node` / `best_match_node` 在普通 RadixCache 中基本相同，但 HiCache 会让它们分出 L1/L2/L3 语义。


### `RadixCache.insert` 与 `cache_finished_req`：什么时候把请求写入 prefix tree

```python
# python/sglang/srt/mem_cache/radix_cache.py:420-472
 420:     def insert(self, params: InsertParams) -> InsertResult:
      # 注：函数定义：`insert` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 421:         if self.disable:
      # 注：分支判断：只有满足 `self.disable` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 422:             return InsertResult(prefix_len=0)
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
 423: 
 424:         key = params.key
      # 注：局部状态绑定：`key` 保存本阶段的中间结果，后续几行通常会立即消费它。
 425:         value = params.value
      # 注：局部状态绑定：`value` 保存本阶段的中间结果，后续几行通常会立即消费它。
 426:         priority = params.priority
      # 注：局部状态绑定：`priority` 保存本阶段的中间结果，后续几行通常会立即消费它。
 427:         chunked = params.chunked
      # 注：局部状态绑定：`chunked` 保存本阶段的中间结果，后续几行通常会立即消费它。
 428: 
 429:         key, value = key.maybe_to_bigram_view(self.is_eagle, value)
      # 注：局部状态绑定：`key, value` 保存本阶段的中间结果，后续几行通常会立即消费它。
 430:         key = key.page_aligned(self.page_size)
      # 注：局部状态绑定：`key` 接住关键调用结果，后续代码会基于它继续装配组件或推进请求。
 431:         if value is not None:
      # 注：分支判断：只有满足 `value is not None` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 432:             value = value[: len(key)]
      # 注：局部状态绑定：`value` 保存本阶段的中间结果，后续几行通常会立即消费它。
 433:         else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
 434:             # Debug/test fallback: use token ids themselves as values.
 435:             value = torch.tensor(key.token_ids[: len(key)], dtype=torch.int64)
      # 注：局部状态绑定：`value` 接住关键调用结果，后续代码会基于它继续装配组件或推进请求。
 436: 
 437:         prefix_len, last_node = self._insert_helper(
      # 注：局部状态绑定：`prefix_len, last_node` 保存本阶段的中间结果，后续几行通常会立即消费它。
 438:             self.root_node, key, value, priority, chunked
 439:         )
 440:         return InsertResult(prefix_len=prefix_len, last_device_node=last_node)
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
 441: 
 442:     def cache_finished_req(self, req: Req, is_insert: bool = True):
      # 注：函数定义：`cache_finished_req` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 443:         """Cache request when it finishes."""
 444:         # In deterministic mode, disable finished request insertion to radix cache
 445:         if self.disable_finished_insert:
      # 注：分支判断：只有满足 `self.disable_finished_insert` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 446:             is_insert = False
      # 注：局部状态绑定：`is_insert` 保存本阶段的中间结果，后续几行通常会立即消费它。
 447: 
 448:         kv_committed_len = req.pop_committed_kv_cache()
      # 注：局部状态绑定：`kv_committed_len` 保存本阶段的中间结果，后续几行通常会立即消费它。
 449:         if self.disable:
      # 注：分支判断：只有满足 `self.disable` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 450:             kv_indices = self.req_to_token_pool.req_to_token[
      # 注：局部状态绑定：`kv_indices` 保存位置/索引映射，通常会连接请求逻辑 token 与 KV 物理槽位。
 451:                 req.req_pool_idx, :kv_committed_len
 452:             ]
 453:             self.token_to_kv_pool_allocator.free(kv_indices)
      # 注：成员函数调用：`self.token_to_kv_pool_allocator.free` 会读取或更新当前对象状态，建议继续查看该方法定义。
 454:             return
      # 注：返回出口：提前结束当前路径，通常表示这个分支已经完成处理或无需继续推进。
 455: 
 456:         token_ids = (req.origin_input_ids + req.output_ids)[:kv_committed_len]
      # 注：局部状态绑定：`token_ids` 保存本阶段的中间结果，后续几行通常会立即消费它。
 457:         kv_indices = self.req_to_token_pool.req_to_token[
      # 注：局部状态绑定：`kv_indices` 保存位置/索引映射，通常会连接请求逻辑 token 与 KV 物理槽位。
 458:             req.req_pool_idx, : len(token_ids)
 459:         ]
 460: 
 461:         radix_key = RadixKey(
      # 注：局部状态绑定：`radix_key` 保存本阶段的中间结果，后续几行通常会立即消费它。
 462:             token_ids, req.extra_key, is_bigram=self.is_eagle
      # 注：局部状态绑定：`token_ids, req.extra_key, is_bigram` 保存请求或 batch 状态，后续调度/执行会继续改写它。
 463:         ).page_aligned(self.page_size)
      # 注：函数/库调用：`page_aligned` 把当前阶段委托给外部 helper 或库实现。
 464:         key_len = len(radix_key)
      # 注：局部状态绑定：`key_len` 保存本阶段的中间结果，后续几行通常会立即消费它。
 465:         values = kv_indices[:key_len].to(dtype=torch.int64, copy=True)
      # 注：局部状态绑定：`values` 保存本阶段的中间结果，后续几行通常会立即消费它。
 466: 
 467:         # Radix Cache takes one ref in memory pool
 468:         if is_insert:
      # 注：分支判断：只有满足 `is_insert` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 469:             priority = getattr(req, "priority", 0) or 0
      # 注：局部状态绑定：`priority` 保存本阶段的中间结果，后续几行通常会立即消费它。
 470:             result = self.insert(
      # 注：局部状态绑定：`result` 保存本阶段的中间结果，后续几行通常会立即消费它。
 471:                 InsertParams(key=radix_key, value=values, priority=priority)
      # 注：局部状态绑定：`InsertParams(key` 保存本阶段的中间结果，后续几行通常会立即消费它。
 472:             )
```

**读这段时抓住：**

- `insert` 返回已有 prefix 长度和总长度，调用者据此知道新增了多少 cache metadata。
- `cache_finished_req` 使用 `origin_input_ids + output_ids[:-1]`，最后一个 token 通常还没有可作为未来 prefix 的完整 KV 后继语义。
- `kv_indices` 来自 req-to-token pool 的逻辑位置到 KV slot 映射，这里把请求生命周期内的 KV 提升为可复用 cache。


## Scheduler 如何“感知 cache”

`SchedulePolicy` 中的 LPM 表示 longest prefix match。等待队列里的请求会先计算 prefix hit，再按更可能复用 KV 的顺序进入 prefill。这就是为什么 prefix cache 不是被动缓存：调度器会主动把“能复用”的请求排到更合适的位置。


### `match_prefix_for_req`：scheduler 把 cache hit 写回 Req

```python
# python/sglang/srt/managers/schedule_policy.py:65-111
  65: 
  66: # Threshold for in-batch prefix cache.
  67: # If a request has a matched prefix length (against existing cache) less than this value,
  68: # the scheduler runs the in-batch prefix caching check for this request.
  69: # If we set it to -1, it means we disable in-batch prefix caching.
  70: IN_BATCH_PREFIX_CACHING_CHECK_THRESHOLD = int(
      # 注：局部状态绑定：`IN_BATCH_PREFIX_CACHING_CHECK_THRESHOLD` 保存请求或 batch 状态，后续调度/执行会继续改写它。
  71:     os.environ.get("IN_BATCH_PREFIX_CACHING_CHECK_THRESHOLD", "32")
      # 注：对象/库方法调用：`os.environ.get` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
  72: )
  73: 
  74: # Threshold for in-batch prefix cache.
  75: # If a request has a matched prefix length (within the waiting queue) larger than this value,
  76: # the scheduler deprioritizes this request
  77: IN_BATCH_PREFIX_CACHING_DEPRIORITIZE_THRESHOLD = int(
      # 注：局部状态绑定：`IN_BATCH_PREFIX_CACHING_DEPRIORITIZE_THRESHOLD` 保存请求或 batch 状态，后续调度/执行会继续改写它。
  78:     os.environ.get("IN_BATCH_PREFIX_CACHING_DEPRIORITIZE_THRESHOLD", "32")
      # 注：对象/库方法调用：`os.environ.get` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
  79: )
  80: 
  81: 
  82: IGNORE_EOS_RESERVE_TOKENS = 1
      # 注：局部状态绑定：`IGNORE_EOS_RESERVE_TOKENS` 保存本阶段的中间结果，后续几行通常会立即消费它。
  83: 
  84: 
  85: def match_prefix_for_req(
      # 注：函数定义：`match_prefix_for_req` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  86:     tree_cache: BasePrefixCache,
  87:     req: Req,
  88:     token_ids: Optional[array[int]] = None,
  89:     *,
  90:     cow_mamba: bool = False,
  91:     include_req: bool = False,
  92: ):
  93:     if token_ids is None:
      # 注：分支判断：只有满足 `token_ids is None` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
  94:         token_ids = req.origin_input_ids + req.output_ids
      # 注：局部状态绑定：`token_ids` 保存本阶段的中间结果，后续几行通常会立即消费它。
  95: 
  96:     match_result = tree_cache.match_prefix(
      # 注：局部状态绑定：`match_result` 接住关键调用结果，后续代码会基于它继续装配组件或推进请求。
  97:         MatchPrefixParams(
      # 注：函数/库调用：`MatchPrefixParams` 把当前阶段委托给外部 helper 或库实现。
  98:             key=RadixKey(token_ids=token_ids, extra_key=req.extra_key),
  99:             cow_mamba=cow_mamba,
 100:             req=req if include_req else None,
 101:         )
 102:     )
 103:     if envs.SGLANG_RADIX_FORCE_MISS.get():
      # 注：分支判断：只有满足 `envs.SGLANG_RADIX_FORCE_MISS.get()` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 104:         match_result = zero_match_result(tree_cache, match_result)
      # 注：局部状态绑定：`match_result` 接住关键调用结果，后续代码会基于它继续装配组件或推进请求。
 105:     (
 106:         req.prefix_indices,
 107:         req.last_node,
 108:         req.last_host_node,
 109:         req.best_match_node,
 110:         req.host_hit_length,
 111:         req.swa_host_hit_length,
```

**读这段时抓住：**

- 这段是 RadixCache 和 Scheduler 的接缝：cache 返回 `MatchResult`，函数把它展开到 `Req` 字段。
- `num_matched_prefix_tokens` 把 device hit 和 host hit 合并成调度视角的 prefix hit。
- `SGLANG_RADIX_FORCE_MISS` 是调试开关，可以强制关闭命中以比较性能/正确性。


### `SchedulePolicy.calc_priority`：prefix cache 会反过来影响谁先 prefill

```python
# python/sglang/srt/managers/schedule_policy.py:145-214
 145:     RANDOM = "random"
      # 注：局部状态绑定：`RANDOM` 保存本阶段的中间结果，后续几行通常会立即消费它。
 146:     ROUTING_KEY = "routing-key"  # prioritize by routing key frequency in running batch
      # 注：局部状态绑定：`ROUTING_KEY` 保存本阶段的中间结果，后续几行通常会立即消费它。
 147: 
 148: 
 149: class SchedulePolicy:
      # 注：类定义：`SchedulePolicy` 是这一段的状态/行为边界；先看字段，再看哪些方法会改字段。
 150:     Policy = Union[CacheAwarePolicy, CacheAgnosticPolicy]
      # 注：局部状态绑定：`Policy` 保存本阶段的中间结果，后续几行通常会立即消费它。
 151: 
 152:     def __init__(
      # 注：函数定义：`__init__` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 153:         self,
 154:         policy: str,
 155:         tree_cache: BasePrefixCache,
 156:         enable_hierarchical_cache: bool,
 157:         enable_priority_scheduling: bool,
 158:         schedule_low_priority_values_first: bool,
 159:     ):
 160:         self.policy = self._validate_and_adjust_policy(policy, tree_cache)
      # 注：成员变量写入：`self.policy` 会留在对象生命周期中，后续方法可能依赖这份状态。
 161:         self.tree_cache = tree_cache
      # 注：成员变量写入：`self.tree_cache` 保存缓存/内存池资源，生命周期会跨越多个请求。
 162:         self.enable_hierarchical_cache = enable_hierarchical_cache
      # 注：成员变量写入：`self.enable_hierarchical_cache` 保存缓存/内存池资源，生命周期会跨越多个请求。
 163:         self.enable_priority_scheduling = enable_priority_scheduling
      # 注：成员变量写入：`self.enable_priority_scheduling` 会留在对象生命周期中，后续方法可能依赖这份状态。
 164:         self.schedule_low_priority_values_first = schedule_low_priority_values_first
      # 注：成员变量写入：`self.schedule_low_priority_values_first` 会留在对象生命周期中，后续方法可能依赖这份状态。
 165:         self.priority_sign = 1 if schedule_low_priority_values_first else -1
      # 注：成员变量写入：`self.priority_sign` 会留在对象生命周期中，后续方法可能依赖这份状态。
 166: 
 167:         # It is used to find the matching prefix for in-batch prefix caching.
 168:         self.waiting_queue_radix_tree = RadixCache.create_simulated()
      # 注：成员变量写入：`self.waiting_queue_radix_tree` 保存队列状态，后续 event loop 会持续消费或填充它。
 169: 
 170:     def calc_priority(
      # 注：函数定义：`calc_priority` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 171:         self, waiting_queue: List[Req], running_batch: Optional[ScheduleBatch] = None
      # 注：字段/变量声明：`self, waiting_queue` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 172:     ) -> None:
 173:         policy = self._determine_active_policy(waiting_queue)
      # 注：局部状态绑定：`policy` 保存本阶段的中间结果，后续几行通常会立即消费它。
 174: 
 175:         # Populate req.num_matched_prefix_tokens at schedule time. Cache-aware policies
 176:         # set it in _compute_prefix_matches; do the same full match for
 177:         # cache-agnostic policies when the radix supports it, so the load
 178:         # snapshot has it. Skip on decode (never prefills).
 179:         if (
      # 注：多行分支开始：完整条件在接下来几行，通常用于组合多个请求参数或运行时状态。
 180:             not isinstance(policy, CacheAwarePolicy)
 181:             and self.tree_cache.supports_fast_match_prefix()
      # 注：成员函数调用：`self.tree_cache.supports_fast_match_prefix` 会读取或更新当前对象状态，建议继续查看该方法定义。
 182:             and get_global_server_args().disaggregation_mode != "decode"
      # 注：函数/库调用：`get_global_server_args` 把当前阶段委托给外部 helper 或库实现。
 183:         ):
 184:             for r in waiting_queue:
      # 注：循环处理：通常是在遍历请求、rank、worker、token 或候选项。
 185:                 match_prefix_for_req(self.tree_cache, r)
      # 注：关键调用：`match_prefix_for_req` - 把 cache 命中结果写回 Req，供排序和 KV 分配使用。
 186: 
 187:         if self.policy == CacheAgnosticPolicy.FCFS:
      # 注：分支判断：只有满足 `self.policy == CacheAgnosticPolicy.FCFS` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 188:             if self.enable_priority_scheduling:
      # 注：分支判断：只有满足 `self.enable_priority_scheduling` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 189:                 SchedulePolicy._sort_by_priority_and_fcfs(
      # 注：对象/库方法调用：`SchedulePolicy._sort_by_priority_and_fcfs` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 190:                     waiting_queue, self.priority_sign
 191:                 )
 192:             return
      # 注：返回出口：提前结束当前路径，通常表示这个分支已经完成处理或无需继续推进。
 193: 
 194:         if isinstance(policy, CacheAwarePolicy):
      # 注：分支判断：只有满足 `isinstance(policy, CacheAwarePolicy)` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 195:             temporary_deprioritized = self._compute_prefix_matches(
      # 注：局部状态绑定：`temporary_deprioritized` 保存本阶段的中间结果，后续几行通常会立即消费它。
 196:                 waiting_queue, policy
 197:             )
 198:             if policy == CacheAwarePolicy.LPM:
      # 注：分支判断：只有满足 `policy == CacheAwarePolicy.LPM` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 199:                 SchedulePolicy._sort_by_longest_prefix(
      # 注：对象/库方法调用：`SchedulePolicy._sort_by_longest_prefix` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 200:                     waiting_queue, temporary_deprioritized
 201:                 )
 202:             elif policy == CacheAwarePolicy.DFS_WEIGHT:
      # 注：补充分支：前面的条件不满足时，再检查 `policy == CacheAwarePolicy.DFS_WEIGHT`。
 203:                 SchedulePolicy._sort_by_dfs_weight(waiting_queue, self.tree_cache)
      # 注：对象/库方法调用：`SchedulePolicy._sort_by_dfs_weight` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 204:             else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
 205:                 raise ValueError(f"Unknown CacheAware Policy: {policy=}")
      # 注：函数/库调用：`ValueError` 把当前阶段委托给外部 helper 或库实现。
 206:         else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
 207:             if policy == CacheAgnosticPolicy.FCFS:
      # 注：分支判断：只有满足 `policy == CacheAgnosticPolicy.FCFS` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 208:                 pass
 209:             elif policy == CacheAgnosticPolicy.LOF:
      # 注：补充分支：前面的条件不满足时，再检查 `policy == CacheAgnosticPolicy.LOF`。
 210:                 SchedulePolicy._sort_by_longest_output(
      # 注：对象/库方法调用：`SchedulePolicy._sort_by_longest_output` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 211:                     waiting_queue,
 212:                     self.enable_priority_scheduling,
 213:                     self.priority_sign,
 214:                 )
```

**读这段时抓住：**

- cache-aware policy 会先计算 prefix matches，再按 LPM 或 DFS weight 排序。
- 队列过长时 LPM 会退化到 FCFS，避免调度成本压过收益。
- 即使是 cache-agnostic policy，只要 radix 支持 fast match，也可能预先填充 prefix 信息供后续 load snapshot 使用。


In [ ]:
show_lines("python/sglang/srt/managers/schedule_policy.py", 65, 130)
show_lines("python/sglang/srt/managers/schedule_policy.py", 170, 235)


## `RadixAttention.forward`：模型层与 backend 的交界

模型文件中普遍会实例化 `RadixAttention`，但真正的 attention 实现由 `get_attn_backend().forward(...)` 决定。这样同一个模型层可以挂 FlashInfer、Triton、FA3、TRT-LLM、MLA 等 backend。


In [ ]:
show_lines("python/sglang/srt/layers/radix_attention.py", 75, 135)


## 小测试：page 对 prefix matching 的影响

KV cache 经常按 page 管理。`RadixKey.match(..., page_size=N)` 会把匹配长度向下对齐到 page 边界。这样能保证返回的 KV span 对底层 page allocator 是合法的。


In [ ]:
x = RadixKey(array("q", [1, 2, 3, 4, 5, 6]))
y = RadixKey(array("q", [1, 2, 3, 4, 9, 9]))
for page in [1, 2, 4]:
    print("page_size", page, "matched", x.match(y, page_size=page))


## 贡献者注意点

- 任何改变 prefix key 语义的 Feature，都要考虑 `extra_key`，否则可能跨租户/跨 LoRA 复用错误 KV。
- 任何改变 KV slot 生命周期的 Feature，都要检查 lock/ref、evict、cache_finished_req/cache_unfinished_req。
- 任何引入 page 粒度的 Feature，都要检查 `page_size > 1` 下是否仍然对齐。
